In [1]:
import sqlite3
import pandas as pd
import numpy as np

In [ ]:
# downloaded from https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/releases/chembl_36/ (the sqlite one) (it's 27 GB unzipped do not download unless you are prepared to deal with that)

In [2]:
con = sqlite3.connect("../chembl_36/chembl_36_sqlite/chembl_36.db")

In [ ]:
cols = "standard_value, standard_type, pchembl_value, mw_freebase, alogp, hba, hbd, psa, rtb, ro3_pass, num_ro5_violations, full_mwt, aromatic_rings, heavy_atoms, qed_weighted, np_likeness_score, bei, sei, le, lle, max_phase, structure_type, chirality"

In [4]:
df = pd.read_sql_query("""SELECT """ + cols + """ FROM activities 
                    JOIN compound_properties ON activities.molregno == compound_properties.molregno
                    JOIN ligand_eff ON activities.activity_id == ligand_eff.activity_id
                    JOIN molecule_dictionary ON activities.molregno == molecule_dictionary.molregno
                    JOIN assays ON activities.assay_id == assays.assay_id""", con)

In [5]:
df.tail(100)

,standard_value,standard_type,pchembl_value,mw_freebase,alogp,hba,hbd,psa,rtb,ro3_pass,...,heavy_atoms,qed_weighted,np_likeness_score,bei,sei,le,lle,max_phase,structure_type,chirality
2191325,8.5,IC50,8.07,361.37,1.73,5.0,2.0,103.54,6.0,N,...,26.0,0.82,-0.43,22.33,7.79,0.42,6.34,NaN,MOL,-1
2191326,1.2,IC50,8.92,361.37,1.58,5.0,2.0,103.54,6.0,N,...,26.0,0.81,-0.25,24.69,8.62,0.47,7.34,NaN,MOL,-1
2191327,3789.0,IC50,5.42,341.37,1.25,5.0,2.0,103.54,6.0,N,...,25.0,0.82,0.00,15.88,5.24,0.30,4.17,NaN,MOL,-1
2191328,11.0,IC50,7.96,373.38,1.73,5.0,2.0,103.54,6.0,N,...,27.0,0.80,0.03,21.32,7.69,0.40,6.23,NaN,MOL,-1
2191329,11.0,IC50,7.96,341.37,1.25,5.0,2.0,103.54,6.0,N,...,25.0,0.82,-0.06,23.31,7.69,0.43,6.71,NaN,MOL,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2191420,0.3,IC50,9.52,365.34,1.14,5.0,2.0,103.54,6.0,N,...,26.0,0.80,-0.09,26.07,9.20,0.50,8.38,NaN,MOL,-1
2191421,740.0,IC50,6.13,435.48,2.54,7.0,2.0,129.32,7.0,N,...,32.0,0.58,-0.68,14.08,4.74,0.26,3.59,NaN,MOL,-1
2191422,72.0,IC50,7.14,455.90,2.89,7.0,2.0,129.32,7.0,N,...,32.0,0.56,-0.62,15.67,5.52,0.30,4.25,NaN,MOL,-1
2191423,59.0,IC50,7.23,436.47,2.13,6.0,3.0,136.40,7.0,N,...,32.0,0.52,-0.37,16.56,5.30,0.31,5.10,NaN,MOL,-1


In [6]:
df = df.dropna(axis=1, how='all')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2191425 entries, 0 to 2191424
Data columns (total 23 columns):
 #   Column              Dtype  
---  ------              -----  
 0   standard_value      float64
 1   standard_type       object 
 2   pchembl_value       float64
 3   mw_freebase         float64
 4   alogp               float64
 5   hba                 float64
 6   hbd                 float64
 7   psa                 float64
 8   rtb                 float64
 9   ro3_pass            object 
 10  num_ro5_violations  float64
 11  full_mwt            float64
 12  aromatic_rings      float64
 13  heavy_atoms         float64
 14  qed_weighted        float64
 15  np_likeness_score   float64
 16  bei                 float64
 17  sei                 float64
 18  le                  float64
 19  lle                 float64
 20  max_phase           float64
 21  structure_type      object 
 22  chirality           int64  
dtypes: float64(19), int64(1), object(3)
memory usage: 384.5+ 

In [8]:
df = df[df['structure_type'] == 'MOL']
df = df.drop("structure_type", axis=1)

In [9]:
df = df[df['max_phase'] != -1]
df['clinical_trial'] = df['max_phase'].notna() 
df = df.drop("max_phase", axis=1)

In [10]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2129486 entries, 0 to 2191424
Data columns (total 22 columns):
 #   Column              Dtype  
---  ------              -----  
 0   standard_value      float64
 1   standard_type       object 
 2   pchembl_value       float64
 3   mw_freebase         float64
 4   alogp               float64
 5   hba                 float64
 6   hbd                 float64
 7   psa                 float64
 8   rtb                 float64
 9   ro3_pass            object 
 10  num_ro5_violations  float64
 11  full_mwt            float64
 12  aromatic_rings      float64
 13  heavy_atoms         float64
 14  qed_weighted        float64
 15  np_likeness_score   float64
 16  bei                 float64
 17  sei                 float64
 18  le                  float64
 19  lle                 float64
 20  chirality           int64  
 21  clinical_trial      bool   
dtypes: bool(1), float64(18), int64(1), object(2)
memory usage: 359.5+ MB


In [11]:
df.to_csv("../chembl_36/chembl_36_cleaned.csv")

In [12]:
## Get a balanced dataset because 
df['clinical_trial'].value_counts()
# Yikes

clinical_trial
False    2021522
True      107964
Name: count, dtype: int64

In [13]:
df_clinical_trial_false = df[df['clinical_trial'] == False]
df_clinical_trial_true = df[df['clinical_trial'] == True]

In [14]:
temp = df_clinical_trial_false.sample(n=107964)

df_balanced = pd.concat([temp, df_clinical_trial_true], axis=0)
df_balanced = df_balanced.sample(frac=1) # shuffle the data just in case
df_balanced

,standard_value,standard_type,pchembl_value,mw_freebase,alogp,hba,hbd,psa,rtb,ro3_pass,...,aromatic_rings,heavy_atoms,qed_weighted,np_likeness_score,bei,sei,le,lle,chirality,clinical_trial
1004681,14.0,IC50,7.85,506.66,4.88,8.0,2.0,97.78,4.0,N,...,4.0,38.0,0.38,-1.08,15.50,8.03,0.28,2.97,1,True
333160,52.0,Ki,7.28,324.45,0.61,6.0,2.0,106.33,3.0,N,...,1.0,19.0,0.85,-0.51,22.45,6.85,0.52,6.67,1,True
229664,2700.0,IC50,5.57,461.40,3.35,7.0,1.0,66.29,6.0,N,...,3.0,31.0,0.57,-1.23,12.07,8.40,0.25,2.22,-1,False
1627441,1830.0,IC50,5.74,384.28,5.23,4.0,1.0,46.51,6.0,N,...,3.0,24.0,0.47,-1.45,14.93,12.34,0.33,0.51,-1,False
221087,30.0,IC50,7.52,459.28,3.95,8.0,1.0,86.98,7.0,N,...,4.0,29.0,0.45,-1.82,16.38,8.65,0.35,3.57,2,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1237431,1737.8,EC50,5.76,530.67,6.96,4.0,1.0,71.16,10.0,N,...,4.0,40.0,0.23,-0.49,10.85,8.09,0.20,-1.20,-1,False
1090810,339.0,IC50,6.47,517.49,4.93,6.0,1.0,109.69,7.0,N,...,4.0,36.0,0.37,-1.23,12.50,5.90,0.25,1.54,-1,False
1178037,12.0,IC50,7.92,592.67,5.32,6.0,1.0,77.59,7.0,N,...,4.0,43.0,0.31,-1.41,13.36,10.21,0.25,2.60,-1,False
1559162,78.0,AC50,7.11,366.50,4.31,3.0,0.0,43.37,0.0,N,...,0.0,27.0,0.60,2.50,19.39,16.39,0.36,2.80,1,True


In [15]:
df_balanced.to_csv("../chembl_36/chembl_36_cleaned_balanced.csv")

In [16]:
df

,standard_value,standard_type,pchembl_value,mw_freebase,alogp,hba,hbd,psa,rtb,ro3_pass,...,aromatic_rings,heavy_atoms,qed_weighted,np_likeness_score,bei,sei,le,lle,chirality,clinical_trial
0,2500.0,IC50,5.60,398.37,4.30,5.0,1.0,100.71,3.0,N,...,4.0,30.0,0.52,-1.35,14.06,5.56,0.26,1.30,-1,False
1,9000.0,IC50,5.05,520.50,5.68,7.0,1.0,119.17,6.0,N,...,5.0,39.0,0.28,-0.87,9.69,4.23,0.18,-0.63,-1,False
2,4000.0,IC50,5.40,543.01,4.27,7.0,2.0,77.93,8.0,N,...,3.0,36.0,0.44,-2.20,9.94,6.93,0.20,1.13,-1,False
3,17000.0,IC50,4.77,468.62,2.32,7.0,2.0,77.93,10.0,N,...,3.0,33.0,0.48,-1.73,10.18,6.12,0.20,2.45,-1,False
4,180.0,IC50,6.75,516.67,4.27,7.0,2.0,77.93,9.0,N,...,4.0,37.0,0.35,-1.85,13.05,8.65,0.25,2.47,-1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2191420,0.3,IC50,9.52,365.34,1.14,5.0,2.0,103.54,6.0,N,...,2.0,26.0,0.80,-0.09,26.07,9.20,0.50,8.38,-1,False
2191421,740.0,IC50,6.13,435.48,2.54,7.0,2.0,129.32,7.0,N,...,3.0,32.0,0.58,-0.68,14.08,4.74,0.26,3.59,-1,False
2191422,72.0,IC50,7.14,455.90,2.89,7.0,2.0,129.32,7.0,N,...,3.0,32.0,0.56,-0.62,15.67,5.52,0.30,4.25,-1,False
2191423,59.0,IC50,7.23,436.47,2.13,6.0,3.0,136.40,7.0,N,...,3.0,32.0,0.52,-0.37,16.56,5.30,0.31,5.10,-1,False
